# Lab 00: How to set up the lab environment

**CPU | API key: `OPENAI_API_KEY` | one small model call**

## Goal

Check in one run that the environment every lab needs is ready: the notebook runs on the repository's own Python,
the packages are at their pinned versions, your key is found and reaches the model, and the shop's data is in place.
Set the repository up first with this lab's README (an Ubuntu VM with VS Code, GitHub Codespaces, or your own machine
with `uv` or `pip`), then run every cell from the top. When the last cell shows ✅ on every required line, you are
ready for Lab 01.

Each cell checks one thing and says what a pass looks like. The optional tools,
environments and keys are reported too, each with the lab that uses it, so you can decide which ones to set up.

## What you will learn

- How to check that a notebook runs on the repository's `.venv` and that its packages match the pins.
- What the config cell at the top of every lab reads from `models.env` and `.env`, and how to change the model in
  one line.
- How to check that your key reaches the model, and what that one call costs.
- Which optional tools, environments and keys exist, and which lab uses each.
- How to read the summary at the end and fix a failed line from the troubleshooting table.

---

## Part A: Read the settings every lab starts from

Every lab opens with this config cell. It finds the repository folder, reads the model settings and finds your key
without printing it. A pass is `key found: yes`, with the model and the reasoning effort from `models.env`. The note
after the checks, "What the config cell does", explains it line by line.

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The cell prints whether it found the key, never the key itself. The next cell prints every setting it read:

In [2]:
# Everything the labs read from models.env (and from .env, where it replaces a setting)
for name, value in settings.items():
    print(f"{name:32} {value}")

MODEL_DEFAULT                    gpt-6-luna
MODEL_STRONG                     gpt-6-luna
REASONING_EFFORT                 none
GEMINI_MODEL                     gemini-3.5-flash-lite
MODEL_COMPARE                    gpt-4o-mini
EMBEDDING_MODEL                  text-embedding-3-small
PRICE_INPUT_PER_MILLION          0.10
PRICE_CACHED_INPUT_PER_MILLION   0.01
PRICE_CACHE_WRITE_PER_MILLION    0.125
PRICE_OUTPUT_PER_MILLION         0.50
PRICE_EMBEDDING_PER_MILLION      0.02
PRICE_CONTAINER_SESSION_USD      0.03


A pass is this list, `MODEL_DEFAULT` first. Each setting has one job:

| Setting | What it is for |
|---|---|
| `MODEL_DEFAULT` | the model almost every cell calls |
| `MODEL_STRONG` | the few tasks that need a more capable model |
| `REASONING_EFFORT` | how much the model reasons before it answers; left empty for a model without that setting |
| `GEMINI_MODEL` | a Google model, called only in cells marked optional (labs 07, 08, 09 and 21), and only with `GEMINI_API_KEY` in `.env` |
| `MODEL_COMPARE` | a second OpenAI model, which lab 21 asks the same question as `MODEL_DEFAULT` |
| `EMBEDDING_MODEL` | the model that turns text into vectors in the retrieval labs |
| `PRICE_INPUT_PER_MILLION` | US dollars per million input tokens sent to `MODEL_DEFAULT` |
| `PRICE_CACHED_INPUT_PER_MILLION` | US dollars per million input tokens served from the prompt cache |
| `PRICE_CACHE_WRITE_PER_MILLION` | US dollars per million input tokens written to the prompt cache |
| `PRICE_OUTPUT_PER_MILLION` | US dollars per million tokens the model writes |
| `PRICE_EMBEDDING_PER_MILLION` | US dollars per million tokens sent to `EMBEDDING_MODEL` |
| `PRICE_CONTAINER_SESSION_USD` | US dollars for one hosted code-interpreter session in lab 17 |

None of them is a secret, so `models.env` is committed to the repository. Your key is in `.env`, which is never
committed.

**To change the model for every lab,** change the `MODEL_DEFAULT=...` line in `models.env` and the prices under it.
**To change it on your machine only,** put the same line in your `.env`. A model without a reasoning setting (such as
`gpt-4o-mini`) also needs `REASONING_EFFORT=` empty in `models.env` itself: an empty value in `.env` does not replace a
setting (step 3 of "What the config cell does"). No notebook names a model in its code, so nothing else changes.

---

## Part B: Check the notebook's Python and the pinned packages

Every saved output in these labs came from the repository's own environment, `.venv`, with the versions pinned in
`pyproject.toml`. First, which Python runs this notebook:

In [3]:
import sys

# A pass is the .venv in the repository folder; the path is shown from that folder
in_venv = Path(sys.prefix).resolve() == (ROOT / ".venv").resolve()
kernel = Path(sys.executable).relative_to(Path(sys.prefix).parent) if in_venv else sys.executable
print("kernel:", kernel, "|", "the repository's .venv" if in_venv else "not the repository's .venv")

kernel: .venv/bin/python | the repository's .venv


A pass is `.venv/bin/python` (on Windows, `.venv\Scripts\python.exe`) and `the repository's .venv`. Any other path is
the Python of another environment: choose the kernel again (README, "Choose the kernel").

Then the Python version, against the range in `pyproject.toml`:

In [4]:
import platform
import tomllib
from packaging.specifiers import SpecifierSet

project = tomllib.loads((ROOT / "pyproject.toml").read_text())["project"]
python_ok = platform.python_version() in SpecifierSet(project["requires-python"])
print("Python", platform.python_version(), "| the repository needs", project["requires-python"],
      "|", "inside the range" if python_ok else "outside the range")

Python 3.13.5 | the repository needs >=3.11,<3.15 | inside the range


A pass is a version inside the range.

Then every pin. The cell reads the pins from `pyproject.toml`, so it still works when a pin changes, and looks up the
version installed in this notebook's environment:

In [5]:
from importlib.metadata import version, PackageNotFoundError

different = []
for requirement in project["dependencies"]:
    name, pin = requirement.split("==")
    name = name.split("[")[0]                 # "openai-agents[viz]" is installed as "openai-agents"
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = "missing"
    if installed != pin:
        different.append(name)
    print(f"{name:28} pin {pin:10} installed {installed}")

print(len(project["dependencies"]), "pinned packages,", len(different), "different")

chromadb                     pin 1.5.9      installed 1.5.9
gradio                       pin 6.31.0     installed 6.31.0
ipykernel                    pin 7.4.0      installed 7.4.0
jupyterlab                   pin 4.6.6      installed 4.6.6
langchain                    pin 1.4.10     installed 1.4.10
langchain-core               pin 1.7.7      installed 1.7.7
langchain-openai             pin 1.7.4      installed 1.7.4
langgraph                    pin 1.2.20     installed 1.2.20
langgraph-checkpoint-sqlite  pin 3.1.3      installed 3.1.3
langsmith                    pin 0.14.17    installed 0.14.17
llama-index-core             pin 0.14.25    installed 0.14.25
llama-index-embeddings-openai pin 0.7.0      installed 0.7.0
llama-index-llms-openai      pin 0.8.4      installed 0.8.4
litellm                      pin 1.109.0    installed 1.109.0
matplotlib                   pin 3.11.2     installed 3.11.2
mcp                          pin 2.3.0      installed 2.3.0
mlflow                       

A pass is `23 pinned packages, 0 different`. A `missing` package, or another version, means the kernel is not the
repository's `.venv`, or the install did not finish. Do not upgrade a pinned package to silence a warning: the labs
were written and run on these versions. The list holds the packages the repository names; `uv` or `pip` installed
what each of them needs as well.

---

## Part C: Check that the key reaches the model

This cell sends one request with your key. A reply shows three things at once: the key is valid, your account can use
`MODEL_DEFAULT`, and this machine reaches the API. Lab 01 explains the call and its response.

⚠️ This cell sends one request billed to your key. It is under 50 tokens, a tiny fraction of a cent.

In [6]:
from openai import OpenAI, omit

client = OpenAI()
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello to a new student in five words."}],
    max_completion_tokens=20,
    # omit leaves the field out of the request when REASONING_EFFORT is empty (a model without reasoning)
    reasoning_effort=EFFORT or omit,
)

tokens_in, tokens_out = reply.usage.prompt_tokens, reply.usage.completion_tokens
cost = (tokens_in * float(settings["PRICE_INPUT_PER_MILLION"])
        + tokens_out * float(settings["PRICE_OUTPUT_PER_MILLION"])) / 1_000_000
print("reply:      ", reply.choices[0].message.content)
print("answered by:", reply.model)
print("tokens:     ", tokens_in, "in,", tokens_out, "out")
print(f"cost:        ${cost:.7f}")

reply:       Hello, welcome to our class today!
answered by: gpt-6-luna
tokens:      16 in, 11 out
cost:        $0.0000071


A pass is any short greeting. Ours was `Hello, welcome to our class today!`: six words, not five, because a model
does not count words exactly. `answered by` names the model that answered, which can be a dated version of the name
in `models.env`. The tokens are what you are billed for, 16 in and 11 out in our run, and the cost
multiplies each count by its price in `models.env`. If the call fails instead, the error names the cause
(troubleshooting table).

---

## Part D: Check the shop's database and documents

Every lab works on one running case, the support desk of Chinook, a music shop, and many checkpoints depend on the
exact contents of its files. First the database, opened read-only (`mode=ro`) so that no lab can change it:

In [7]:
import sqlite3

db_path = ROOT / "data" / "chinook.db"
db = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, check_same_thread=False)
customers = db.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
invoices = db.execute("SELECT COUNT(*) FROM invoices").fetchone()[0]
db.close()
print("chinook.db:", customers, "customers,", invoices, "invoices")

chinook.db: 59 customers, 412 invoices


A pass is 59 customers and 412 invoices. Other counts mean the file was replaced or changed (troubleshooting table).

Then the shop's policies and support tickets, which the retrieval labs search:

In [8]:
corpus = sorted((ROOT / "data" / "corpus").glob("*.md"))
policies = [f for f in corpus if f.name.startswith("policy_")]
tickets = [f for f in corpus if f.name.startswith("ticket_")]
print("data/corpus:", len(corpus), "files,", len(policies), "policies,", len(tickets), "tickets")
print("first and last:", corpus[0].name, "...", corpus[-1].name)

data/corpus: 36 files, 20 policies, 16 tickets
first and last: policy_accounts.md ... ticket_16_overpayment_note.md


A pass is 36 files: 20 policies and 16 tickets.

Then the one PDF in `data/documents/`, which lab 07's file search reads:

In [9]:
guide = ROOT / "data" / "documents" / "AI_Governance_Frameworks_Reference_Guide.pdf"
print("data/documents:", guide.name, "found" if guide.is_file() else "missing")

data/documents: AI_Governance_Frameworks_Reference_Guide.pdf found


A pass is `found`.

---

## Part E: Check the optional tools, environments and keys

Most labs need nothing from this part. Each line says what you have and which lab uses it, so a missing piece only
means you have not set it up; it never fails a check. The README says how to install each one.

**Graphviz.** Lab 07 draws an agent's structure with `dot`, a program from Graphviz:

In [10]:
import shutil
import subprocess

dot_found = shutil.which("dot") is not None
if dot_found:
    print(subprocess.run(["dot", "-V"], capture_output=True, text=True).stderr.strip())
else:
    print("dot: not found (optional; lab 07 draws agents with it)")

dot - graphviz version 15.1.1 (20260805.0921)


A pass is a version line like ours, `dot - graphviz version 15.1.1 (20260805.0921)`. Without Graphviz the line
says `not found`; in lab 07 only the drawing cells need it.

**Docker.** Lab 17's optional last part runs model-written code in a container on your machine; the rest of lab 17
runs without Docker. `docker info` answers only when Docker is installed and running:

In [11]:
if shutil.which("docker") is None:
    docker_ok = False
    print("docker: not installed (optional; lab 17's container runner)")
else:
    result = subprocess.run(["docker", "info", "--format", "{{.ServerVersion}}"], capture_output=True, text=True)
    docker_ok = result.returncode == 0
    print("docker server:", result.stdout.strip() if docker_ok else result.stderr.strip())

docker server: 29.9.2


A pass is a server version, ours `29.9.2`. If Docker is installed but not running, the line shows the
`failed to connect to the docker API` error from the troubleshooting table.

**Node.js.** Lab 06 opens its MCP server in the MCP Inspector, a Node.js program that `npx` downloads on first use.
The Inspector needs Node.js 22.19 or newer:

In [12]:
if shutil.which("npx") is None:
    node_ok = False
    print("node: not installed (optional; lab 06's MCP Inspector)")
else:
    node_version = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
    node_ok = node_version.lstrip("v") in SpecifierSet(">=22.19")
    print("node", node_version, "|", "22.19 or newer" if node_ok else "older than 22.19, which the MCP Inspector needs")

node v26.7.0 | 22.19 or newer


A pass is a version of 22.19 or newer; ours was `v26.7.0`. Without Node.js, lab 06's two Inspector cells print one
line and skip.

**Labs with their own environment.** Three labs need other versions of packages the main environment pins (an older
`openai`, for example), so each keeps its own `.venv` in its folder. The cell looks for each one:

In [13]:
OWN_ENVIRONMENTS = {
    "lab 08": "08_Google_ADK",
    "bonus B4": "bonus/B4_CrewAI_Crew",
    "bonus B5": "bonus/B5_Microsoft_Agent_Framework",
}
own_ready = {}
for lab, folder in OWN_ENVIRONMENTS.items():
    own_ready[lab] = (ROOT / folder / ".venv").is_dir()
    print(f"{lab:9} {folder:36} {'set up' if own_ready[lab] else 'optional, not set up'}")

lab 08    08_Google_ADK                        optional, not set up
bonus B4  bonus/B4_CrewAI_Crew                 optional, not set up
bonus B5  bonus/B5_Microsoft_Agent_Framework   optional, not set up


In our run none of the three was set up. These lines never fail: you create a lab's own environment when you do that
lab (README, "Labs with their own environment").

**Optional keys.** A Google key runs the cells marked optional on a Gemini model, and two bonus labs use services that
need an account: Pinecone (B6) and LangSmith (B7). Their lines are in `.env.example`. The cell says whether each key is
set, never its value:

In [14]:
OPTIONAL_KEYS = {
    "GEMINI_API_KEY": "optional cells in labs 07, 08, 09 and 21",
    "PINECONE_API_KEY": "bonus B6",
    "LANGSMITH_API_KEY": "bonus B7",
}
key_set = {}
for name, used_by in OPTIONAL_KEYS.items():
    key_set[name] = bool(os.environ.get(name) or local.get(name))
    print(f"{name:18} {'set' if key_set[name] else 'optional, not set':18} {used_by}")

GEMINI_API_KEY     set                optional cells in labs 07, 08, 09 and 21
PINECONE_API_KEY   set                bonus B6
LANGSMITH_API_KEY  set                bonus B7


Every core lab runs without these three keys.

---

## Check that you are ready

One line per check: the required ones pass or fail, and the optional ones say what is set up. If you skipped a cell,
this cell stops with a `NameError`; run that cell, then this one.

In [15]:
required = {
    "the kernel is the repository's .venv": in_venv,
    "Python inside the range, every package at its pin": python_ok and not different,
    "models.env read, the key found": bool(MODEL and os.environ["OPENAI_API_KEY"]),
    "the model answers": bool(reply.choices[0].message.content),
    "chinook.db: 59 customers, 412 invoices": (customers, invoices) == (59, 412),
    "data/corpus: 20 policies, 16 tickets": (len(policies), len(tickets)) == (20, 16),
    "data/documents: the governance guide": guide.is_file(),
}
optional = {
    "Graphviz dot (lab 07)": dot_found,
    "Docker running (lab 17)": docker_ok,
    "Node.js 22.19 or newer (lab 06)": node_ok,
    **{f"own environment, {lab}": ready for lab, ready in own_ready.items()},
    **{f"{name} ({used_by})": key_set[name] for name, used_by in OPTIONAL_KEYS.items()},
}

print("Python", platform.python_version(), "|", reply.model)
for name, passed in required.items():
    print("✅" if passed else "❌", name)
print("\nOptional:")
for name, ready in optional.items():
    print("  ✅" if ready else "  --", name if ready else name + ": optional, not set up")

Python 3.13.5 | gpt-6-luna
✅ the kernel is the repository's .venv
✅ Python inside the range, every package at its pin
✅ models.env read, the key found
✅ the model answers
✅ chinook.db: 59 customers, 412 invoices
✅ data/corpus: 20 policies, 16 tickets
✅ data/documents: the governance guide

Optional:
  ✅ Graphviz dot (lab 07)
  ✅ Docker running (lab 17)
  ✅ Node.js 22.19 or newer (lab 06)
  -- own environment, lab 08: optional, not set up
  -- own environment, bonus B4: optional, not set up
  -- own environment, bonus B5: optional, not set up
  ✅ GEMINI_API_KEY (optional cells in labs 07, 08, 09 and 21)
  ✅ PINECONE_API_KEY (bonus B6)
  ✅ LANGSMITH_API_KEY (bonus B7)


In our run every required line passed. Graphviz, Docker, Node.js and the three optional keys were available; none of the three labs' own environments was set up. With every required line passing you are ready for Lab 01. A failed
line names the check; find it in its part above and in the troubleshooting table below, which lists each message, its
cause and the fix.

### Checkpoint

| Check | A pass | Our run |
|---|---|---|
| Part A, config | `key found: yes`, and the model from `models.env` | `key found: yes`, `gpt-6-luna`, reasoning effort `none` |
| Part B, Python and packages | `.venv/bin/python`; a Python inside `>=3.11,<3.15`; `0 different` | `.venv/bin/python`; Python 3.13.5; 23 pinned packages, 0 different |
| Part C, model call | any short reply, and the model that answered | `Hello, welcome to our class today!` from `gpt-6-luna`, 16 tokens in and 11 out |
| Part D, data | 59 customers, 412 invoices; 36 files (20 policies, 16 tickets); the governance guide `found` | 59 customers, 412 invoices; 36 files (20 policies, 16 tickets); `found` |
| Part E, optional | a version line, or a line saying what is not set up | Graphviz 15.1.1, Docker 29.9.2, Node.js v26.7.0 and the three optional keys; the environments of labs 08, B4 and B5 not set up |

## What the config cell does

Every lab opens with the cell from Part A, and later labs run it without this explanation. In order, it:

1. **Finds the repository folder.** `[Path.cwd(), *Path.cwd().parents]` lists the notebook's folder and every folder
   above it, nearest first, and `next(...)` returns the first one that holds `models.env`. So the same cell works in
   any lab folder. When no folder holds `models.env` (the notebook was copied out of the repository), `next` has
   nothing to return and stops with `StopIteration`.
2. **Reads the settings.** `dotenv_values` reads a file of `NAME=value` lines into a dictionary: `settings` from
   `models.env`, and `local` from your `.env` (empty when there is no `.env`).
3. **Lets `.env` replace a setting.** `settings.update(...)` copies a value from `.env` only when its name is already
   in `models.env` and the value is not empty. So a `MODEL_DEFAULT=...` line in `.env` changes the model on your
   machine only, an empty line such as `REASONING_EFFORT=` in `.env` changes nothing, and the key never enters
   `settings`.
4. **Finds the key.** `os.environ.setdefault` keeps an `OPENAI_API_KEY` that is already in the environment (a
   Codespaces secret, for example) and otherwise takes the one from `.env`. The OpenAI library reads the key from the
   environment, so no later cell passes it or prints it.
5. **Stops early without a key.** `assert` stops the notebook here with its message, instead of three cells later
   with an authentication error.
6. **Names the two settings every call uses.** `MODEL` is `MODEL_DEFAULT`. `EFFORT` is `REASONING_EFFORT`, and
   `or None` turns an empty value into `None`, which Part C leaves out of the request with `omit`.

## What you can do now

- Check that a notebook runs on the repository's `.venv` and that every package matches its pin.
- Say what the config cell finds and prints, and why it can stop with `StopIteration` or an `AssertionError`.
- Change the model for every lab in `models.env`, or for your machine only in `.env` (a model without reasoning also
  needs `REASONING_EFFORT=` empty in `models.env`).
- Check that your key reaches the model, and read what one call cost: ours was 16 tokens in and 11 out, $0.0000071.
- Tell which optional tools, environments and keys you have, and which lab needs each.
- Find a failed line in the troubleshooting table and fix it.

## Troubleshooting

Each message below is shown as the error prints it, with `...` where a long part is cut. The setup problems that
come before the notebook (installing `uv`, a blocked `pip`) are in the README.

| Where, and the message | Cause | Fix |
|---|---|---|
| Config cell: `ModuleNotFoundError: No module named 'dotenv'` | the notebook runs on a Python without the repository's packages | choose the repository's `.venv` as the kernel (README, "Choose the kernel"), then run from the top |
| Config cell: `StopIteration` | the notebook is not inside the repository folder (it was copied, downloaded alone or opened from elsewhere) | open it from the cloned or unzipped repository, where `models.env` sits |
| Config cell: `AssertionError: Put OPENAI_API_KEY in .env at the repository folder (copy .env.example).` | no `.env` beside `models.env`, or its `OPENAI_API_KEY=` line is empty | copy `.env.example` to `.env`, paste the key after `OPENAI_API_KEY=`, then run the config cell again |
| Part B: `kernel: .../08_Google_ADK/.venv/bin/python \| not the repository's .venv` | the kernel is not the `.venv` in the repository folder: another lab's own `.venv` (here lab 08's), a system Python, or a conda environment or venv under another name | choose the `.venv` in the repository folder as the kernel |
| Part B: most packages `missing`, and `openai` at `2.54.0` | the same: the kernel is lab 08's own environment, not the repository's | choose the `.venv` in the repository folder as the kernel |
| Part C: ``OpenAIError: Missing credentials. Please pass an `api_key`, ...`` | the config cell stopped without a key, and the cells after it were run anyway | fix the key (the `AssertionError` row), then run from the config cell |
| Part C: `AuthenticationError: Error code: 401 - ... Incorrect API key provided ...` | the key is wrong, cut short or revoked, or an old `OPENAI_API_KEY` in the environment wins over `.env` | paste the key again; if it is set in the environment (a Codespaces secret, a shell profile), fix it there |
| Part C: `NotFoundError: Error code: 404 - The model ... does not exist or you do not have access to it.` | `MODEL_DEFAULT` is misspelt, or your account cannot use that model | check the line in `models.env`, and any `MODEL_DEFAULT` line in `.env` |
| Part C: `BadRequestError: Error code: 400 - Unrecognized request argument supplied: reasoning_effort` | `REASONING_EFFORT` is set for a model without reasoning | empty the `REASONING_EFFORT=` line in `models.env` itself (an empty line in `.env` does not replace the setting), and remove a `REASONING_EFFORT` line from `.env` |
| Part C: `reply:` is empty, and the last cell shows ❌ on `the model answers` | the model spent the 20-token ceiling (`max_completion_tokens=20`) on reasoning, so no text was left: `REASONING_EFFORT` is above `none`, or empty on a model that reasons | set `REASONING_EFFORT=none` in `models.env` and remove a `REASONING_EFFORT` line from `.env`, or use a model without reasoning with `REASONING_EFFORT=` empty in `models.env` |
| Part D: `OperationalError: unable to open database file` | `data/chinook.db` is missing or was moved | restore it from the repository: `git checkout -- data/chinook.db`, or unzip the ZIP again |
| Part D: `IndexError: list index out of range`, or ❌ on the `chinook.db`, `data/corpus` or `data/documents` line of the last cell | files in `data/` were changed, replaced or removed | restore them: `git checkout -- data/`, or unzip the ZIP again; then run Part D and the last cell again |
| Part E: `dot: not found`; in lab 07, `ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH` | Graphviz is not installed, or not on the `PATH` | install Graphviz (README, "Optional tools"), then restart the kernel |
| Part E: `failed to connect to the docker API at unix:///...; check if the path is correct and if the daemon is running` | Docker is installed but not running, or `DOCKER_HOST` points to a socket that does not exist | start Docker Desktop, or on Linux `sudo systemctl start docker`; unset a stale `DOCKER_HOST`; then run the cell again |
| Part E: `node v20.11.0 \| older than 22.19, which the MCP Inspector needs` | the installed Node.js is too old for the MCP Inspector | install a current Node.js (README, "Optional tools"), then restart the kernel |

## Resources

- [uv: working on projects](https://docs.astral.sh/uv/guides/projects/): `uv sync`, `uv run` and the lock file
- [python-dotenv](https://pypi.org/project/python-dotenv/): `dotenv_values` and the `.env` file format
- [Python `sqlite3`: SQLite URIs](https://docs.python.org/3/library/sqlite3.html#how-to-work-with-sqlite-uris): the
  read-only `mode=ro` address in Part D
- [Node.js: download](https://nodejs.org/en/download): installers and package managers for each system
- [OpenAI: models](https://developers.openai.com/api/docs/models) and
  [pricing](https://developers.openai.com/api/docs/pricing): the names and prices you can put in `models.env`

Next: [Lab 01: How to call a model and read the response](../01_Model_Call/README.md). Back to
[the lab list](../README.md).